# Q2. Unsupervised Learning — Customer Segmentation with K-Means and PCA

**Data source:** GitHub raw file at `https://raw.githubusercontent.com/ritikadalela/ml-assessment-Ritika-Dalela/74af08cf34b00fde293ca57387128a8e5d244abe/Data/q2_customers.csv`

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_style("whitegrid")
RANDOM_STATE = 42

## 1. Data Preparation

In [ ]:
RAW_BASE_URL = "https://raw.githubusercontent.com/ritikadalela/ml-assessment-Ritika-Dalela/74af08cf34b00fde293ca57387128a8e5d244abe/Data"

def load_csv_from_github(filename):
    url = f"{RAW_BASE_URL}/{filename}"
    return pd.read_csv(url)

df = load_csv_from_github("q2_customers.csv")
print("Loaded from:", f"{RAW_BASE_URL}/q2_customers.csv")
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
df.head()


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

scaled_df = pd.DataFrame(X_scaled, columns=df.columns)
scaled_df.head()

Scaling is essential before K-Means because the algorithm is distance-based. Features such as `annual_spend` operate on a much larger numeric range than features such as `visits_per_month`, so without scaling, high-magnitude variables would dominate the cluster assignment even if they are not the most meaningful behavioural drivers.

## 2. Choosing K — Elbow Method

In [ ]:
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method for Choosing K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS / Inertia')
plt.xticks(range(1, 11))
plt.show()

pd.DataFrame({'K': range(1, 11), 'WCSS': wcss})

In [ ]:
silhouette_rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_rows.append({'K': k, 'silhouette_score': silhouette_score(X_scaled, labels)})

pd.DataFrame(silhouette_rows)

The elbow is most visible around **K = 4**: inertia drops sharply from 1 to 4 clusters and then begins to flatten. Although the silhouette score is highest at K = 2, choosing 4 gives a more useful business segmentation because it preserves meaningful customer variety while still being close to the elbow.

## 3. K-Means Clustering

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

centroids_df = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=df.drop(columns='cluster').columns
).round(2)

print("Cluster counts:")
print(df['cluster'].value_counts().sort_index())
print("\nCluster centroids:")
centroids_df

In [ ]:
cluster_profile = df.groupby('cluster').mean(numeric_only=True).round(2)
cluster_profile

### Business interpretation of the clusters

- **Cluster 0**: younger customers with low annual spend, smaller baskets, and frequent recent visits — likely value-driven regular shoppers.
- **Cluster 1**: older, very high-spending customers who buy across many categories but have long gaps since their last visit — likely premium but at-risk customers.
- **Cluster 2**: middle-of-the-road customers across most metrics — a broad mainstream segment.
- **Cluster 3**: older, high-spending, broad-category shoppers with more recent activity than Cluster 1 — likely loyal premium customers.

## 4. Dimensionality Reduction with PCA

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

explained_variance_df = pd.DataFrame({
    'Principal Component': ['PC1', 'PC2'],
    'Explained Variance Ratio': pca.explained_variance_ratio_
})

loadings_df = pd.DataFrame(
    pca.components_.T,
    index=df.drop(columns='cluster').columns,
    columns=['PC1', 'PC2']
).round(3)

print("Explained variance ratio:")
display(explained_variance_df)

print("Feature loadings:")
loadings_df

PC1 explains the large majority of the variation and appears to capture an axis moving from **frequent/recent, lower-value shopping** toward **older, bigger-basket, higher-spend, wider-category purchasing**. PC2 is dominated by **`days_since_last_visit`**, so it mainly separates customers by recency and potential churn risk.

## 5. Cluster Visualisation

In [ ]:
pca_plot_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_plot_df['cluster'] = df['cluster']

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_plot_df, x='PC1', y='PC2', hue='cluster', palette='tab10', s=60)
plt.title('Customer Segments Visualised in PCA Space')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()